In [4]:
import os
from pathlib import Path

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
env_path = Path("../.env")
if env_path.exists():
    for raw in env_path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip().strip("\"'")
        os.environ.setdefault(key, value)

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in ../.env or environment before running notebook.")

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen3-0.6B")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [6]:
model.to("cuda:1")

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [7]:
messages = "What is your name?"
inputs = tokenizer(messages, return_tensors="pt")

In [8]:
inputs.to(model.device)

{'input_ids': tensor([[3838,  374,  697,  829,   30]], device='cuda:1'), 'attention_mask': tensor([[1, 1, 1, 1, 1]], device='cuda:1')}

In [9]:
outputs = model.generate(**inputs, max_new_tokens=20)
print(outputs)
print(tokenizer.decode(outputs[0]))

tensor([[3838,  374,  697,  829,   30, 3555,  374,  697, 3476,   30, 3555,  374,
          697, 1887, 5795,   30, 3555,  374,  697, 1482, 3728,   30, 3555,  374,
          697]], device='cuda:1')
What is your name? What is your role? What is your main goal? What is your current location? What is your


In [10]:
from datasets import load_dataset

ds = load_dataset("vicgalle/alpaca-gpt4")

In [11]:
ds['train'][0]

{'instruction': 'Give three tips for staying healthy.',
 'input': '',
 'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for 

In [12]:
def to_messages(example):
    user_text = example['instruction'].strip()
    if example['input'].strip():
        user_text += "\n\n" + example['input'].strip()
    return {
        "messages":[
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": example['output']},
        ]
    }
ds_messages = ds.map(to_messages, remove_columns=ds['train'].column_names)

In [13]:
# list(ds_messages['train']['messages'])

In [14]:
def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=True,
        add_generation_prompt=False,
        return_tensors="pt",
    )
    return {'text':text}

inputs = ds_messages.map(apply_chat_template)


In [18]:
inputs['train']
# lengths = [len(text['input_ids'][0]) for text in inputs['train']['text']]
# print(lengths)

Dataset({
    features: ['messages', 'text'],
    num_rows: 52002
})

In [ ]:
# 将 ds["train"] 划分为 train / val（例如 98% / 2%）
split_ds = ds["train"].train_test_split(test_size=0.02, seed=42)

train_ds = split_ds["train"]
val_ds = split_ds["test"]

# 保存为 JSONL
train_ds.to_json("../data/train.jsonl", force_ascii=False)
val_ds.to_json("../data/val.jsonl", force_ascii=False)

print(f"train size: {len(train_ds)}")
print(f"val size: {len(val_ds)}")
print("saved: train.jsonl, val.jsonl")

Creating json from Arrow format:   0%|          | 0/51 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

train size: 50961
val size: 1041
saved: train.jsonl, val.jsonl


In [24]:
ds = load_dataset("../data/alpaca_gpt4", split="validation")

In [25]:
ds

Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 1041
})